# Validate against buoy observations

Compare **offshore hindcast** (closest goal node), **bulk metamodel**, **spectral metamodel**, and **dynamic SnapWave** to buoy records.

**Prerequisites**

1. Fitted models — `06_Validate_Metamodel.ipynb` (`mm.fit_all`)
2. Dynamic run for the same period — `08_Dynamic_Cases_Generation.ipynb`
3. Observations GeoJSON — `03_Generate_Observations_Configuration.ipynb` → `inputs/Observations/<Case>_observations.json`
4. Processed buoy file(s) under `inputs/Observations/processed/` (you build these in the notebook below)

Use the **same `period`** for buoy data, reconstruction, and dynamic output.

Comparisons request **`hs`, `tp`, `dir`, and `spr`**. Variables missing from the buoy
file or any model are skipped automatically.

In [1]:
import time

import matplotlib.pyplot as plt
import xarray as xr
from utils.config import SIMULATIONS, resolve
from utils import validation as val
from utils import metamodel as mm
from utils import onshore_sites as obs

## Configuration

Auto-resolves every buoy with real spectral observations under
`inputs/Observations/processed/spectra/` (currently 3 of 35 — see notebook 11) and
validates all of them together, pooled across every year they have data for. Point
`spectral_buoy_dir` elsewhere, or hand-pick `buoy_instruments`, to validate a
different set.

In [2]:
period_label = (
    "2013, 2017-2023"  # years these 18 nearshore buoys have real observations
)
years = [
    2013,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
]  # not contiguous — see notebook 11
offshore_name = "WHACS"
mda_n = 450  # must match a fitted mda_candidates entry
pca_variance = 0.99

sites = [
    "buoy_Bengello_Buoy",
    "buoy_BoomerangA2_Buoy",
    "buoy_BoomerangB2_Buoy",
    "buoy_Broulee_Buoy",
    "buoy_Broulee2_Buoy",
    "buoy_Collaroy4_Buoy",
    "buoy_Collaroy5_Buoy",
    "buoy_Deeban_Buoy",
    "buoy_Deeban2_Buoy",
    "buoy_Eden_Buoy",
    "buoy_FairyMeadow_Buoy",
    "buoy_Farquhar_Buoy",
    "buoy_Figure8Pools_Buoy",
    "buoy_Merimbula_Buoy",
    "buoy_Merimbula2_Buoy",
    "buoy_OldBarA_Buoy",
    "buoy_Stockton_Buoy",
    "buoy_Woonona_Buoy",
]

# Processed observation NetCDF (bulk — no raw spectra for these) per SnapWave site id
buoy_raw_files = {
    "buoy_Bengello_Buoy": resolve("inputs/Observations/processed/Bengello_Buoy.nc"),
    "buoy_BoomerangA2_Buoy": resolve(
        "inputs/Observations/processed/BoomerangA2_Buoy.nc"
    ),
    "buoy_BoomerangB2_Buoy": resolve(
        "inputs/Observations/processed/BoomerangB2_Buoy.nc"
    ),
    "buoy_Broulee_Buoy": resolve("inputs/Observations/processed/Broulee_Buoy.nc"),
    "buoy_Broulee2_Buoy": resolve("inputs/Observations/processed/Broulee2_Buoy.nc"),
    "buoy_Collaroy4_Buoy": resolve("inputs/Observations/processed/Collaroy4_Buoy.nc"),
    "buoy_Collaroy5_Buoy": resolve("inputs/Observations/processed/Collaroy5_Buoy.nc"),
    "buoy_Deeban_Buoy": resolve("inputs/Observations/processed/Deeban_Buoy.nc"),
    "buoy_Deeban2_Buoy": resolve("inputs/Observations/processed/Deeban2_Buoy.nc"),
    "buoy_Eden_Buoy": resolve("inputs/Observations/processed/Eden_Buoy.nc"),
    "buoy_FairyMeadow_Buoy": resolve(
        "inputs/Observations/processed/FairyMeadow_Buoy.nc"
    ),
    "buoy_Farquhar_Buoy": resolve("inputs/Observations/processed/Farquhar_Buoy.nc"),
    "buoy_Figure8Pools_Buoy": resolve(
        "inputs/Observations/processed/Figure8Pools_Buoy.nc"
    ),
    "buoy_Merimbula_Buoy": resolve("inputs/Observations/processed/Merimbula_Buoy.nc"),
    "buoy_Merimbula2_Buoy": resolve("inputs/Observations/processed/Merimbula2_Buoy.nc"),
    "buoy_OldBarA_Buoy": resolve("inputs/Observations/processed/OldBarA_Buoy.nc"),
    "buoy_Stockton_Buoy": resolve("inputs/Observations/processed/Stockton_Buoy.nc"),
    "buoy_Woonona_Buoy": resolve("inputs/Observations/processed/Woonona_Buoy.nc"),
}

requested_variables = ["hs", "tp", "dir", "spr"]

dynamic_path = (
    resolve(SIMULATIONS["dynamic_cases"])
    / f"Dynamic_Cases_{offshore_name}_{period_label}.nc"
)

print("Observations JSON:", obs.geojson_path())
print(f"period         : {period_label}")
print(f"mda_n          : {mda_n}")
print(f"sites          : {sites}")
print(f"requested vars : {requested_variables}")
print(f"dynamic path   : {dynamic_path}")
print(f"dynamic exists : {dynamic_path.is_file()}")

Observations JSON: /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/inputs/Observations/NewSouthWales_observations.json
period         : 2013, 2017-2023
mda_n          : 450
sites          : ['buoy_Bengello_Buoy', 'buoy_BoomerangA2_Buoy', 'buoy_BoomerangB2_Buoy', 'buoy_Broulee_Buoy', 'buoy_Broulee2_Buoy', 'buoy_Collaroy4_Buoy', 'buoy_Collaroy5_Buoy', 'buoy_Deeban_Buoy', 'buoy_Deeban2_Buoy', 'buoy_Eden_Buoy', 'buoy_FairyMeadow_Buoy', 'buoy_Farquhar_Buoy', 'buoy_Figure8Pools_Buoy', 'buoy_Merimbula_Buoy', 'buoy_Merimbula2_Buoy', 'buoy_OldBarA_Buoy', 'buoy_Stockton_Buoy', 'buoy_Woonona_Buoy']
requested vars : ['hs', 'tp', 'dir', 'spr']
dynamic path   : /hyglowdata/NewSouthWales/SnapWave_CASES_Dynamic/Dynamic_Cases_WHACS_2013, 2017-2023.nc
dynamic exists : False


## 1 · Check models

In [3]:
mm.require_models_all(mda_n, pca_variance=pca_variance)

## 2 · Process buoy observations

Load each processed file, rename variables to HyGloW names (`hs`, `tp`, `dir`, `spr`),
slice to `period`, and build a validation dataset on `(time, sites)` with CF-style
attributes. Site metadata (lon/lat/depth) comes from the observations GeoJSON.

In [4]:
# CF-style attributes for validation plots
_BUOY_VAR_ATTRS = {
    "hs": {"standard_name": "sea_surface_wave_significant_height", "units": "m"},
    "tp": {"standard_name": "sea_surface_wave_period", "units": "s"},
    "dir": {"standard_name": "sea_surface_wave_from_direction", "units": "degree"},
    "spr": {
        "standard_name": "sea_surface_wave_directional_spread",
        "units": "degree",
    },
}

# Years aren't contiguous (see `years` above) but slicing the buoy files
# themselves is free — no reconstruction involved — so a single wide slice
# is simplest; each buoy's own real data still only lands where it exists.
buoy_period = slice(str(years[0]), str(years[-1]))

chunks: list[xr.Dataset] = []
for site_id in sites:
    raw = xr.open_dataset(buoy_raw_files[site_id]).sel(time=buoy_period)

    if "Efth" in raw or "efth" in raw:
        spec_name = "Efth" if "Efth" in raw else "efth"
        stats = raw[spec_name].spec.stats(["hs", "tp", "dm", "dspr"]).load()
        site_ds = stats.rename({"dm": "dir", "dspr": "spr"})
    else:
        rename = {}
        if "dspr" in raw.data_vars and "spr" not in raw.data_vars:
            rename["dspr"] = "spr"
        if rename:
            raw = raw.rename(rename)
        vars_out = {}
        for name in requested_variables:
            if name in raw.data_vars:
                vars_out[name] = raw[name]
            elif name == "dir" and "dm" in raw.data_vars:
                vars_out["dir"] = raw["dm"]
        if "hs" not in vars_out:
            raise ValueError(f"{site_id}: processed buoy file must contain hs or Efth")
        site_ds = xr.Dataset(vars_out)

    site_ds = site_ds.expand_dims(sites=[site_id]).transpose("time", "sites")
    for var, attrs in _BUOY_VAR_ATTRS.items():
        if var in site_ds:
            site_ds[var].attrs.update(attrs)
    chunks.append(obs.attach_buoy_site_meta(site_ds, site_id))

buoy = xr.concat(chunks, dim="sites") if len(chunks) > 1 else chunks[0]
buoy

<xarray.Dataset> Size: 78MB
Dimensions:         (time: 133179, sites: 18)
Coordinates:
  * time            (time) datetime64[ns] 1MB 2013-01-01T03:59:59.999996416 ....
  * sites           (sites) object 144B 'buoy_Bengello_Buoy' ... 'buoy_Woonon...
    site_source     (sites) <U4 288B 'buoy' 'buoy' 'buoy' ... 'buoy' 'buoy'
    site_label      (sites) <U17 1kB 'Bengello_Buoy' ... 'Woonona_Buoy'
    site_longitude  (sites) float64 144B 150.2 152.5 152.6 ... 152.6 151.8 150.9
    site_latitude   (sites) float64 144B -35.88 -32.34 -32.35 ... -32.9 -34.35
    site_depth      (sites) float64 144B -12.81 -13.29 -32.85 ... -13.16 -13.62
Data variables:
    hs              (time, sites) float64 19MB nan nan nan nan ... nan nan nan
    tp              (time, sites) float64 19MB nan nan nan nan ... nan nan nan
    dir             (time, sites) float64 19MB nan nan nan nan ... nan nan nan
    spr             (time, sites) float64 19MB nan nan nan nan ... nan nan nan

## 3 · Reconstruct metamodel, load dynamic, and closest-goal offshore hindcast

Metamodel bulk/spectral reconstructions, dynamic SnapWave, and bulk offshore forcing
at each buoy's **closest goal hindcast node** (all sources in the case JSON —
currently WHACS only).

In [5]:
# One reconstruct() call per year (not a single slice — see `years` in
# Configuration) so we never trigger fresh reconstruction for 2014-2016,
# which these buoys have no data for and were never cached. Every year here
# already has a cached bulk+spectral reconstruction, so this is all cache
# hits, not new computation.
timings: dict[str, float] = {}

t0 = time.perf_counter()
bulk = xr.concat(
    [
        mm.reconstruct(
            str(y),
            mda_n,
            spectral=False,
            offshore_name=offshore_name,
            sites=sites,
            pca_variance=pca_variance,
        )
        for y in years
    ],
    dim="time",
)
timings["Bulk metamodel"] = time.perf_counter() - t0

t0 = time.perf_counter()
spectral = xr.concat(
    [
        mm.reconstruct(
            str(y),
            mda_n,
            spectral=True,
            offshore_name=offshore_name,
            sites=sites,
            pca_variance=pca_variance,
        )
        for y in years
    ],
    dim="time",
)
timings["Spectral metamodel"] = time.perf_counter() - t0
# Both timings cover every year in the loop above; per this cell's opening
# comment they're normally all cache hits, so near-zero is expected, not a
# bug. Offshore hindcast below is a plain lookup, not a reconstruction, so
# it has no timing entry.

# if not dynamic_path.is_file():
#     raise FileNotFoundError(
#         f"Dynamic output not found: {dynamic_path}\n"
#         "Run 08_Dynamic_Cases_Generation for this period or change `period`."
#     )

# dynamic = xr.open_dataset(dynamic_path).sel(time=buoy_period)
# site_ids = mm.resolve_reconstruction_sites(dynamic, sites)
# dynamic = dynamic.sel(sites=site_ids).load()

offshore_hindcast_by_year = [
    val.load_offshore_hindcast_at_buoys(str(y), sites, variables=requested_variables)
    for y in years
]
offshore_hindcast = {
    label: xr.concat([d[0][label] for d in offshore_hindcast_by_year], dim="time")
    for label in offshore_hindcast_by_year[0][0]
}
closest_goals = offshore_hindcast_by_year[0][1]
closest_goals

estimates = {
    **offshore_hindcast,
    "Bulk metamodel": bulk,
    "Spectral metamodel": spectral,
    # "Dynamic SnapWave": dynamic,
}

variables = val.resolve_comparison_variables(
    buoy,
    *estimates.values(),
    variables=requested_variables,
)
print(f"Comparing: {variables}")

Comparing: ['hs', 'tp', 'dir', 'spr']


## 4 · Metrics vs buoy

In [6]:
results = val.compare_all(
    buoy,
    estimates,
    sites=sites,
    variables=variables,
    reference_label="Buoy",
    pca_variance=pca_variance,
)

summary = val.metrics_summary(results)
summary

,pca_variances,group,site_source,site_label,variable,label,n,bias,rmse,r2,estimate_label,reference_label,model
0,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_Bengello_Buoy,buoy,waves,hs,Hs,22577,0.951662,1.117914,-0.793726,WHACS (closest goal),Buoy,WHACS (closest goal)
1,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_Bengello_Buoy,buoy,waves,tp,Tp,22577,-0.504670,2.368282,-0.026598,WHACS (closest goal),Buoy,WHACS (closest goal)
2,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_Bengello_Buoy,buoy,waves,dir,Dir,22577,10.433747,49.407730,-13.177957,WHACS (closest goal),Buoy,WHACS (closest goal)
3,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_Bengello_Buoy,buoy,waves,spr,Spr,22577,12.672102,17.836072,-0.827045,WHACS (closest goal),Buoy,WHACS (closest goal)
4,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_BoomerangA2_Buoy,buoy,waves,hs,Hs,7221,0.740474,0.915227,-0.526817,WHACS (closest goal),Buoy,WHACS (closest goal)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_Stockton_Buoy,buoy,waves,spr,Spr,11754,-3.323877,10.880435,-0.036911,Spectral metamodel,Buoy,Spectral metamodel
212,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_Woonona_Buoy,buoy,waves,hs,Hs,3522,0.325329,0.448019,0.124555,Spectral metamodel,Buoy,Spectral metamodel
213,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_Woonona_Buoy,buoy,waves,tp,Tp,3522,-1.756723,3.271639,-1.345141,Spectral metamodel,Buoy,Spectral metamodel
214,"{'hs': 0.99, 'dir': 0.99, 'spr': 0.99}",buoy_Woonona_Buoy,buoy,waves,dir,Dir,3522,13.759628,27.615537,-1.877212,Spectral metamodel,Buoy,Spectral metamodel


## 4b · Save summary

In [7]:
summary_out = summary.copy()
summary_out.insert(0, "period", period_label)
summary_out.insert(1, "mda_candidate", mda_n)
summary_out.insert(2, "offshore_name", offshore_name)
summary_out["reconstruct_seconds"] = summary_out["model"].map(timings)

out_path = val.buoy_validation_path("onshore", period_label)
out_path.parent.mkdir(parents=True, exist_ok=True)
summary_out.to_csv(out_path, index=False)
print(f"Wrote {out_path}")

Wrote /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/onshore_summary_2013-2017-2023.csv


## 5 · Scatter plots

In [8]:
# for site in sites:
#     for name, result in results.items():
#         result.plot_scatter(
#             site,
#             variables=variables,
#             estimate_label=name,
#             reference_label="Buoy",
#             cmap="Reds",
#         )
#         plt.suptitle(f"{name} vs buoy — {site} — {period_label}")
#         plt.show()

## 6 · Time series

Line plots of buoy observations vs each model over the validation period (direction as scatter).

In [9]:
# for site in sites:
#     val.plot_buoy_timeseries(buoy, estimates, site, variables=variables)
#     plt.suptitle(f"Buoy validation — {site} — {period_label}")
#     plt.show()

## 7 · Save per-buoy summary PNG (GUI)

Density scatter (top row, buoy vs. spectral metamodel) + one wide Hs/Tp time series (bottom row, cropped to when the buoy actually has data) — one combined PNG per buoy, shown as-is in the GUI's Buoy Validation tab when you click that buoy's star on the map.

In [12]:
sites

['buoy_Bengello_Buoy',
 'buoy_BoomerangA2_Buoy',
 'buoy_BoomerangB2_Buoy',
 'buoy_Broulee_Buoy',
 'buoy_Broulee2_Buoy',
 'buoy_Collaroy4_Buoy',
 'buoy_Collaroy5_Buoy',
 'buoy_Deeban_Buoy',
 'buoy_Deeban2_Buoy',
 'buoy_Eden_Buoy',
 'buoy_FairyMeadow_Buoy',
 'buoy_Farquhar_Buoy',
 'buoy_Figure8Pools_Buoy',
 'buoy_Merimbula_Buoy',
 'buoy_Merimbula2_Buoy',
 'buoy_OldBarA_Buoy',
 'buoy_Stockton_Buoy',
 'buoy_Woonona_Buoy']

In [11]:
for site in sites:
    fig = val.plot_buoy_summary(
        buoy,
        estimates,
        site,
        variables=variables,
        reference_label="Buoy",
        period_label=period_label,
        cmap="Reds",
    )
    out_path = val.buoy_summary_figure_path(site)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"saved {out_path}")

saved /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/figures/buoy_Bengello_Buoy.png
saved /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/figures/buoy_BoomerangA2_Buoy.png
saved /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/figures/buoy_BoomerangB2_Buoy.png
saved /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/figures/buoy_Broulee_Buoy.png
saved /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/figures/buoy_Broulee2_Buoy.png
saved /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/figures/buoy_Collaroy4_Buoy.png
saved /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/figures/buoy_Collaroy5_Buoy.png
saved /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/figures/buoy_Deeban_Buoy.png
saved /home/tausiaj/GitHub-GeoOcean/HyGloW/NewSouthWales/outputs/buoy_validation/figures/buoy_Deeba